# ReGRID model analysis
### Sensitivity analysis tool

- For evluating economic impacts of data center addition on the power system.
- This notebook depends on output files generated by the following notebook:
  - model_DC.ipynb

In [1]:
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [2]:
# Model definition

TERM = '1y'
STEP = 6
VER = 'v7'
YEAR = '2050'
BOUNDARY = 'japan'
PATHWAY = 'C1'
NUCLEAR_AVAILABILITY = 1
POTENTIAL_AVAILABILITY= 50
POTENTIAL_CASE = 'base'
COST_CASE = 'moderate'
DC_SCENARIO = 'NoDC'
DC_STRATEGY = 'NoDC'

In [3]:
model_name = f"ReGRID2_{VER}_{YEAR}_{PATHWAY}_{TERM}_{STEP}_{BOUNDARY}_n{NUCLEAR_AVAILABILITY}_p{POTENTIAL_AVAILABILITY}_{POTENTIAL_CASE}_{COST_CASE}_{DC_SCENARIO}_{DC_STRATEGY}"
print(model_name)

ReGRID2_v7_2050_C1_1y_6_japan_n1_p50_base_moderate_NoDC_NoDC


In [ ]:
# Data for analysis

# Result
dual = pd.read_csv('output/%s/dual_%s.csv' % (model_name,model_name), index_col=0, dtype={'region':str, 'time':str})
dual = dual.replace('_', '', regex=True) #Remove blank
dual = dual.replace(' ', '', regex=True) #Remove blank

# Technology dataset
P = pd.read_csv(f'input/technology/technology_{PATHWAY}.csv', index_col=0)
P = P[P['year'] == int(YEAR)]

# Region list
region_list = pd.read_csv('input/region.csv', index_col='code', dtype={'code':str, 'DSN':str})
REGION = region_list.index.to_list() # All nodes

# Data center demand profile
DC_demand = pd.read_csv('input/%sh/datacenter_100MW.csv' % (STEP), index_col=0).sum()
DC_demand.index = pd.to_numeric(DC_demand.index)
DC_demand = DC_demand.sort_index()

# JPY -> USD
USD = 107 # TTM in 2020

In [5]:
# Chart paramter
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Arial'
plt.rcParams['xtick.direction'] = 'out'
plt.rcParams['ytick.direction'] = 'out'
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['xtick.major.size'] = 2.5
plt.rcParams['ytick.major.size'] = 2.5
plt.rcParams['font.size'] = 7
plt.rcParams['xtick.labelsize'] = 7
plt.rcParams['ytick.labelsize'] = 7
plt.rcParams['axes.linewidth'] = 0

MAX_WIDTH = 180 / 25.4 # inch in 180mm

In [42]:
# Draw map

def map_plot(map_data, map_label, cmap, map_min, map_max, map_title):

    # Boundary
    municipal_shapes = list(shpreader.Reader("input/analysis/shp/1741municipalities/1741municipalities.shp").geometries())
    japan_shapes = list(shpreader.Reader("input/analysis/shp/Japan/Japan.shp").geometries())

    # Colormap
    if map_max == "max":
        color_max = map_data.max()
    else:
        color_max = map_max

    if map_min == "min":
        color_min = map_data.min()
    else:
        color_min = map_min

    norm = mcolors.Normalize(vmin=color_min, vmax=color_max)

    # Map
    fig = plt.figure(figsize=(MAX_WIDTH*0.5, MAX_WIDTH*0.5*1.25), dpi=3000)
    ax = plt.axes(projection=ccrs.PlateCarree())

    # Regional polygon
    for n, city in enumerate(municipal_shapes):
        value = min(map_data.iloc[n], color_max)
        value = max(value, color_min)
        polygon_color = cmap(norm(value)) if value > 0 else "#eee"
        ax.add_geometries(city, ccrs.PlateCarree(), edgecolor=(0.5,0.5,0.5,1), facecolor=polygon_color, linewidth = 0.0, zorder=1)

    area_cood = [126.6, 147.0, 25.6, 45.9]
    ax.set_extent(area_cood, crs=ccrs.PlateCarree())
    ax.add_geometries(japan_shapes, ccrs.PlateCarree(), edgecolor=(0.1,0.1,0.1,1), facecolor=(0,0,0,0), linewidth = 0.05, zorder=2)

    # Gridline
    gl = ax.gridlines(draw_labels=True, linewidth=0.1, color='gray', alpha=0.6, linestyle='--')
    gl.top_labels = True
    gl.right_labels = False
    gl.bottom_labels = False
    gl.left_labels = True
    gl.xlabel_style = {'fontsize': 5, 'color': "gray"}
    gl.ylabel_style = {'fontsize': 5, 'color': "gray"}
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xpadding = 1
    gl.ypadding = 1
    
    # Draw
    heat_map = ax.imshow([[color_min],[color_max]], cmap=cmap, norm=norm)
    cbar = fig.colorbar(heat_map, shrink=0.83, aspect=40, pad=0.02, orientation='horizontal')
    cbar.set_label(map_label, fontsize=5, labelpad=3)
    cbar.ax.tick_params(labelsize=5, pad=1)
    cbar.ax.set_xscale('linear')
    
    plt.axis('off')
    if map_title is not None:
        plt.savefig("output/%s/%s.jpg" % (model_name,map_title), bbox_inches='tight', pad_inches=0, transparent=False, dpi=3000)
        plt.savefig("output/%s/%s.eps" % (model_name,map_title), bbox_inches='tight', pad_inches=0, dpi=300)
    plt.show()
    plt.clf()

In [7]:
# Integration cost of a data center

DC_cost = dual[(dual['id'] == 'rel')].set_index('region')['value'] / USD # million JPY/DC -> million USD/DC
DC_cost = -1 * DC_cost # reduced cost due to demand reduction -> additional cost due to demand addition
DC_cost.index = pd.to_numeric(DC_cost.index)
DC_cost = DC_cost.sort_index()
DC_demand = DC_demand.sort_index()
unit_DC_cost = DC_cost*1000000 / DC_demand # million USD/DC -> USD/MWh

DC_cost_summary = pd.DataFrame(index=DC_cost.index)
DC_cost_summary['system_cost'] = DC_cost
DC_cost_summary['unit_cost'] = unit_DC_cost
DC_cost_summary.to_csv("output/%s/mrc_%s.csv" % (model_name,model_name))

print('min:',unit_DC_cost.min())
print('max:',unit_DC_cost.max())
print('2.5%:',unit_DC_cost.quantile(0.025))
print('97.5%:',unit_DC_cost.quantile(0.975))

min: 38.43435425239337
max: 124.43122385463644
2.5%: 54.891832907405416
97.5%: 74.42735485805942


In [ ]:
# Map

map_plot(unit_DC_cost, 'Integration cost of data center (USD MWh$^{-1}$)', cm.RdYlBu_r, unit_DC_cost.quantile(0.025), unit_DC_cost.quantile(0.975), f'map_location_unit_cost_DC_95_{model_name}')
# map_plot(DC_cost, 'Additional system costs (Million US$)', cm.Blues, DC_cost.quantile(0.025), DC_cost.quantile(0.975), f'map_location_cost_DC_95_{model_name}')

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


<Figure size 640x480 with 0 Axes>